In [2]:
from __future__ import annotations
import os 
import uuid
import chromadb
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader

In [30]:
# config
PERSIST_DIR = "./chroma_multimodal_local_dataloader"
COLLECTION_NAME = "pets_local_dataloader"

DOG_IMAGE = ["images/dog2.png", "images/dog3.png"]
CAT_IMAGE = ["images/cat1.png","images/cat3.png"]

In [31]:
# เตรียม path
from importlib.resources import path


all_image_paths = DOG_IMAGE + CAT_IMAGE
labels = ["dog"] * len(DOG_IMAGE) + ["cat"] * len(CAT_IMAGE)
doc_ids = [str(uuid.uuid4()) for _ in all_image_paths]

# สร้าง document description
document_desc = [f"This is a photo of a {lbl}" for lbl in labels]

metadatas = [
    {
        "label": lbl,
        "file_path": os.path.basename(path),
        "description": desc
    }
    for lbl, path, desc in zip(labels, all_image_paths, document_desc)
]

In [32]:
client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenCLIPEmbeddingFunction(),
    data_loader=ImageLoader(),
    metadata={"hnsw:space": "cosine"} # เพราะสนใจความหมายมากกว่าระยะห่าง
)

In [33]:
from PIL import Image

for path in all_image_paths:
    try:
        with Image.open(path) as img:
            img.verify() # ตรวจสอบว่าไฟล์รูปภาพสมบูรณ์ไหม
    except Exception as e:
        print(f"พบปัญหาที่ไฟล์: {path} -> {e}")

In [34]:
# เพิ่มข้อมูลเข้าไปใน db
collection.add(
    ids=doc_ids,
    uris=all_image_paths,
    metadatas=metadatas
)

print(f"indexed {len(doc_ids)} images into collection '{COLLECTION_NAME}'.")

indexed 4 images into collection 'pets_local_dataloader'.


In [40]:
# Step 5 - ทดสอบ Query ด้วยข้อความ
query_text = "dog"
res = collection.query(
  query_texts=[query_text], # <--- ค้นหาด้วยคำว่า "dog"
  n_results=3,
  include=["metadatas","documents" ,"distances"]
)
# แสดงผลลัพธ์
print(f"\\n=== Query: '{query_text}' ===")
for rank, (meta, dist) in enumerate(zip(res["metadatas"][0], res["distances"][0]), start=1):
  print(f"#{rank} -> {meta['file_path']} ({meta['label']}) distance={dist:.4f}")

\n=== Query: 'dog' ===
#1 -> dog2.png (dog) distance=0.7209
#2 -> dog3.png (dog) distance=0.7235
#3 -> cat1.png (cat) distance=0.8066
